# 01 — Data and measures  [DATA]

Corpus, codebook, outcome definitions, and predictor sources for the five-predictor negative-binomial model of `racialised_rate`. Consolidated from `archive/supervisor_run_2026-08-16.ipynb` and `archive/rate_total_analysis.ipynb` — see `../ANALYSIS_LOG.md` for the full history.

## Contents

- **1.1** Corpus construction
- **1.2** Hierarchical codebook
- **1.3** Descriptive counts
- **1.4** Outcome definitions and the `rate_total = rate_social × share_social` identity
- **1.5** Distributions of both outcomes
- **1.6** Scatter across constituencies (rate_social vs. rate_total)
- **1.7** Decomposition table (per-predictor additivity check)
- **1.8** Denominator note
- **1.9** Outcome decision
- **1.10** Predictor sources and construction

### 1.1 Corpus construction  [DATA — SETTLED]

5,529 verified German swimming-pool POIs (reconciled from an administrative registry and a Google Maps scrape, spatially + fuzzy-address matched), 1,265,806 text reviews after filtering to verified pools and text-only reviews (`gmaps_data/master_corpus_final.parquet`). Every review row joins to exactly one verified POI (100%). See `gmaps_data/README.md` for the full ETL pipeline.

### 1.2 Hierarchical codebook  [DATA — SETTLED]

Single zero-shot LLM classifier, applied in stages: **`social`** (binary — does the review reference people/staff/atmosphere, vs. facilities/pricing only) → **`racialised`** (binary, `social=='yes'` only — attributes anything to an ethnic/national/cultural/religious group) → four content dimensions (`theme`, `context`, `approach`, `connotation`; `racialised=='yes'` only).

**This hierarchy is load-bearing:** a review cannot be `racialised` without first being `social`, and cannot carry a content dimension without first being `racialised`. Every downstream rate, model, and robustness check in notebooks 02–04 respects this — `rate_social = racialised_reviews / social_reviews` is defined only within the `social` subset.

### 1.3 Descriptive counts  [DATA — SETTLED]

| grain | n | % of corpus |
|---|---|---|
| entire corpus | 1,265,806 | — |
| `social == 'yes'` | 504,072 | 39.8% |
| `racialised == 'yes'` | 7,768 | 1.54% of social, 0.61% of corpus |
| pools with ≥1 racialised review | 1,802 | — |

Constituency-level: 299 constituencies, all with ≥4 pools and ≥678 text reviews (no true zero-coverage constituency). Reliability bar for constituency-level rates set at ≥10 racialised reviews → **247/299 (82.6%) reliable**.

*(Source: `outputs/B1_prevalence_summary.csv`, `outputs/D1_*_properties.csv`.)*

### 1.4 Outcome definitions and the additivity identity  [DATA — SETTLED]

- `rate_social = racialised_reviews / social_reviews` — the primary outcome (matches the research question: among reviews that engage with people/atmosphere, how many racialise).
- `rate_total = racialised_reviews / total_text_reviews` — reported as a sensitivity check (§4.1).
- `share_social = social_reviews / total_text_reviews`.

By construction, `rate_total = rate_social × share_social`. §1.7 checks whether this identity is recovered when all three quantities are independently fit as NB models rather than computed as raw ratios.

### 1.5 Distributions of both outcomes  [DATA — SETTLED]

| | rate_social | rate_total |
|---|---|---|
| mean | 0.0158 | 0.0063 |
| median | 0.0122 | 0.0048 |
| max | 0.0858 | 0.0375 |
| skew | 2.47 | — |

Both are right-skewed count-rate outcomes over a small denominator — the reason the modelling strategy (notebook 02) moves off OLS. *(Source: `outputs/D1_rate_properties.csv`, `outputs/rate_total/C1_rate_total_describe.csv`.)*

### 1.6 Scatter across constituencies  [DATA — SETTLED]

Pearson r = 0.984, Spearman ρ = 0.980 between `rate_social` and `rate_total` across all 299 constituencies. **Part of this correlation is mechanical** — both outcomes share the same numerator (`racialised_reviews`) — so the high correlation on its own doesn't confirm the two outcomes behave identically as a function of the predictors; §1.7 and §4.1 check that separately. *(Source: `rate_total_analysis.ipynb` C1, moved to `archive/`.)*

### 1.7 Decomposition table  [DATA — SETTLED]

Per-predictor NB coefficients (log scale) for `rate_social`, `share_social`, their sum, and the full 5-predictor `rate_total` fit, plus `beta_share` as a % of `beta_social`:

| predictor | β_social | β_share | sum | β_total (full 5) | diff | β_share as % of β_social |
|---|---|---|---|---|---|---|
| foreign_population_pct_c | 0.0406 | −0.0060 | 0.0346 | 0.0356 | −0.0010 | **−14.8%** |
| log_population_density_c | 0.0836 | 0.0200 | 0.1036 | 0.0970 | 0.0066 | +23.9% |
| sgb_ii_recipients_per_1k_c | 0.0059 | 0.0009 | 0.0067 | 0.0066 | 0.0001 | +14.4% |
| is_east | 0.2299 | −0.0078 | 0.2221 | 0.2383 | −0.0162 | −3.4% |
| in_split_city | −0.2781 | −0.0011 | −0.2792 | −0.2555 | −0.0237 | +0.4% |

Max absolute deviation (`sum` vs. `β_total`): **0.024** — additivity holds cleanly across three independently-fit NB models, not just by definition of the raw counts. `foreign_population_pct_c` is the one predictor where `β_share` runs **opposite-signed** to `β_social` — areas with more foreign population write relatively *fewer* social reviews even as those social reviews racialise more, a genuine compositional effect worth flagging, though small (−14.8% of `β_social`). *(Source: `outputs/rate_total/C7c_additivity_check.csv`, `%` column added here.)*

### 1.8 Denominator note  [DATA — SETTLED]

Because racialised reviews sit inside their own denominator (`social_reviews`), the measured rate rises slightly less than the underlying quantity would — by roughly the size of the rate itself. At the mean rate (1.6%) this is a ~1.6% understatement; at the observed maximum (8.6%) it's a ~8.6% understatement. Not large enough to change any conclusion, but worth stating once rather than re-deriving per section.

### 1.9 Outcome decision  [DATA — SETTLED]

`rate_social` and `rate_total` are substantively equivalent: all five predictors carry the same sign under both, and every CI overlaps between the two (§4.1). `rate_social` is the primary outcome because it matches the research question directly (racialisation conditional on engaging socially); `rate_total` is retained as a sensitivity check, not a competing model.

### 1.10 Predictor sources and construction  [DATA — SETTLED]

Socioeconomic/political predictors: `socio-economic_data/socio_political_data.csv` (see `README_data_processing.md` for the cleaning pipeline; 43/66 raw columns are state-repeated in Berlin/Hamburg/Bremen, flagged and re-tested with/without those constituencies throughout §4.1/§2.6). Constituency geometry: `btw25_geometrie_wahlkreise_shp` (EPSG:25832 for all distance/area calculations). `is_east` and `in_split_city` are constructed flags (five-state East bloc; constituencies sharing identical values on six socioeconomic columns, indicating a shared city-level figure rather than an independent measurement).